# KhaledDSML 20 — Specialized research & report export
## الارتباطات الرتبية ونقاط التغيّر وتقارير Word/Excel

**Source development:** 0.10.0rc1 (GitHub), not published as PyPI stable. All samples below synthetic; statistical estimates come from code. / بيانات اصطناعية ونتائج تُحسب فعليًا. Use Python 3.10–3.13.

In [ ]:
%pip install -q "git+https://github.com/noble2013/KhaledDSML.git@main" "statsmodels>=0.14" "python-docx>=1.1" "openpyxl>=3.1"

## 1 — Polychoric | ارتباط المتغيرين الكامنين

Ordinal integer classes from correlated latent Gaussian observations, not arbitrary continuous observations. / قياسات رتبية وفق نموذج طبيعي كامن.

In [ ]:
import numpy as np
import pandas as pd
from khaled_dsml import (
  polychoric_latent_correlation,
  williams_steiger_overlapping_correlations,
  conover_iman_posthoc, pelt_mean_shift,
  ddm_binary_error_monitor, adwin_bounded_mean_monitor,
  save_analysis_xlsx,save_analysis_docx,
  display_analysis_dashboard,
)
rng=np.random.default_rng(71)
latent=rng.multivariate_normal([0,0],[[1,.60],[.60,1]],size=800)
ordinal_x=np.digitize(latent[:,0],[-.7,.25])
ordinal_y=np.digitize(latent[:,1],[-.35,.75])
poly=polychoric_latent_correlation(ordinal_x,ordinal_y)
display_analysis_dashboard(poly,theme="amethyst")

## 2 — Williams–Steiger | مقارنة ارتباطين داخل العينة نفسها

Trivariate iid measurement is essential. / ثلاثة متغيرات للأفراد أنفسهم.

In [ ]:
x=rng.normal(size=150)
y=.7*x+rng.normal(size=150)*.5
z=.2*x+rng.normal(size=150)*.8
display_analysis_dashboard(
    williams_steiger_overlapping_correlations(x,y,z),
    theme="ocean"
)

## 3 — Conover–Iman | المقارنات البعدية للرتب

This implementation is tie-free. / ترفض هذه الدالة الرتب المتساوية بدلاً من تقريب غير مثبت.

In [ ]:
groups=[rng.normal(loc=g*.4,size=17) for g in range(4)]
display_analysis_dashboard(conover_iman_posthoc(*groups),theme="aurora")

## 4 — Exact PELT | تقسيم السلسلة بأقل كلفة

Penalty is prespecified and there are no p-values. / تقديرات مواقع التغير تتأثر بالعقوبة ولا تمثل قيم دلالة.

In [ ]:
values=np.r_[rng.normal(0,.2,size=70),rng.normal(3,.2,size=70)]
result=pelt_mean_shift(values,penalty=15,min_segment_size=8)
display_analysis_dashboard(result,theme="amethyst")
print("Detected zero-based change starts:", result["change_indices"])

## 5 — DDM and ADWIN-style | مراقبة الانجراف

DDM is a heuristic binary-error monitor; this bounded ADWIN-style window is uncompressed and uses a conservative finite-horizon Hoeffding comparison, not canonical optimized ADWIN2.

In [ ]:
errors=np.r_[rng.binomial(1,.03,80),rng.binomial(1,.65,100)]
display_analysis_dashboard(ddm_binary_error_monitor(errors),theme="ocean")
stream=np.r_[rng.uniform(0,.05,90),rng.uniform(.95,1,90)]
display_analysis_dashboard(
    adwin_bounded_mean_monitor(stream,delta=.05,min_subwindow=12),
    theme="aurora"
)

## 6 — Real editable DOCX and XLSX | تصدير ملفات Word وExcel حقيقية

The files are written in this notebook environment. On Windows use a local path. / تحفظ الملفات على جهاز التنفيذ دون إرسال البيانات إلى خادم خارجي.

In [ ]:
save_analysis_docx(result, "KhaledDSML_Research_Analysis.docx")
save_analysis_xlsx(result, "KhaledDSML_Research_Analysis.xlsx")
print("Saved DOCX and XLSX scientific reports")

## Verification boundaries | حدود الاعتماد

- Polychoric requires ordinal categories and latent bivariate Gaussian assumptions.
- Steiger requires **matched triples**; Conover requires independent groups and no tied pooled ranks.
- PELT is an exact optimizer of penalized SSE, not a calibrated significance test.
- DDM alarms are heuristic; bounded-window Hoeffding assumptions are much stronger than mere sequence ordering.
- Formatted exports do not validate research methodology.

**Current catalogue:** 245/256 callable mappings, 11 missing, and *no claim of full scientific sign-off*. / لا يُعتمد اكتمال الكتالوج بمجرد وجود اسم دالة.